In [ ]:
import torch
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, Module
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
import matplotlib.pyplot as plt

In [ ]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).float()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [ ]:
class CustomModel(Module):
    def __init__(self, input_size: int, output_size) -> None:
        super().__init__()
        self.net = Sequential(
            Linear(input_size, 32),
            ReLU(),
            Linear(32,16),
            ReLU(),
            Linear(16, output_size)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [ ]:
def train_model_reg(features, targets, kf, batch, epochs, criterion):
    r2_scores = []
    mse_scores = []
    rmse_scores = []
    folds_loss = []
    for i, (train_index, test_index) in enumerate(kf.split(features)):
        print(f"Treinando modelo ----- Fold {i + 1}")
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch)
        model = CustomModel(10, 1)
        optmizer = Adam(model.parameters(), lr=0.001)
        epoch_loss = []
        for i_epochs in range(epochs):    
            model.train()
            batches_loss = []
            for x, y in train_loader:
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                batches_loss.append(loss.item())
                optmizer.step()

            epoch_loss.append(np.mean(batches_loss))

        folds_loss.append(epoch_loss)
        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float()
            y_true = targets[test_index]
            result = model(x_test_tensor).cpu().numpy()
            r2_s = r2_score(y_true, result)
            rmse_s = root_mean_squared_error(y_true, result)
            mse_s = mean_squared_error(y_true, result)
            r2_scores.append(r2_s)
            rmse_scores.append(rmse_s)
            mse_scores.append(mse_s)
            print(f"R^2: {r2_s}")
            print(f"RMSE: {rmse_s}")
            print(f"MSE: {mse_s}")

    print(f"\nResultado modelo ----- R^2: {np.mean(r2_scores)}, RMSE: {np.mean(rmse_scores)}, MSE: {np.mean(mse_scores)}")

    return folds_loss

In [ ]:
def plot_loss(folds_loss):
    for fold_l in folds_loss:
        fig, ax = plt.subplots(figsize=(7, 3))
        x = np.linspace(0, len(fold_l), len(fold_l))
        ax.plot(x, fold_l)
        ax.set_xlabel("Épocas")
        ax.set_ylabel("MSE Loss")
        ax.set_ylim(0, 5000)
        fig.show()


In [ ]:
dataset = load_diabetes()
features = dataset.data
targets = np.expand_dims(dataset.target, axis=1)

In [ ]:
criterion = MSELoss()
max_epochs = 500
batch = 12
kf = KFold(n_splits=5, shuffle=True)

In [ ]:
folds_loss = train_model_reg(features, targets, kf, batch, max_epochs, criterion)

In [ ]:
plot_loss(folds_loss)